These are some DS, which are / have methods in them which are by characteristic, thread-safe.

# Lists (only the append() method)

the append() method in lists is thread-safe because the underlying 
C function that executes it holds the GIL for its entire duration, 
making it uninterruptible.

Went on a whole different rabbithole while studying about this... 

But a lot of this is necessary. 

### 1. How does python code get executed? 
-> There's actually 2 ways...
i. if it's a library function, maybe a method from a class that's already included in python... in that case this is the pipeline

python -> bytecode -> C function corresponding to the bytecode call gets executed (if it's Cpython) -> machine code

note: how to check what python you're using

```
import platform
print(platform.python_implementation())
```

ii. If it's a custom method / function...

python -> bytecode -> machine code

This is why some methods in python are pretty fast, because it's running C under the hood.

### 2. what's bytecode?
-> Whatever you see in a .pyc file.. it's an intermediate between python and machine code, looks kind of like assembly...

### 3. The GIL / locks can release after any one bytecode instruction, unless you've programmed it specifically not to release a lock...


### 4. Lists are mutable, this is a key hint at realizing that most of the operations with lists are not going to be thread-safe out of the box...

? What are mutable data structures / data types?
-> Mutable ds are those that can be modified in-place. That's it.

e.g. l = [1, 2, 3]
l.append(4) # this is appending 4 to the SAME list, it's not creating a brand-new list..

list, dicts etc are mutable..

Similarly, immutable ds are those that you CANNOT modify in-place... Whenever you modify it, you're basically reassigning a new value, with the same variable name.

e.g.

a = 5
print(id(a))  # e.g. 140732648
a += 1
print(id(a))  # e.g. 140732664 <- the int var itself changes address, meaning it's simply been reassigned..

ints, floats, strings, tuples are immutable...


# Queues

Queues come in a range of different styles. In Python, we have the option to define three different types of queues from the native queue module. These are normal Queues,
LifoQueues, and PriorityQueues.

Queues are by default, thread-safe. all operations included in queues are.

### 1. FIFO Queues

Regular queues that you and I are familiar with.

In [ ]:
import threading
from queue import Queue
import time

def mySubscriber(queue: Queue):
    while not queue.empty():
        item = queue.get()
        if item is None:
            break
        print("{} removed {} from the queue".format(threading.current_thread(), item))
        queue.task_done()
        time.sleep(1)

myQueue = Queue()
for i in range(10):
    myQueue.put(i)

print("Queue Populated")

threads = []
for i in range(4):
    thread = threading.Thread(target=mySubscriber, args=(myQueue,))
    thread.start()
    threads.append(thread)

for thread in threads:
    thread.join()

### 2. LIFO Queues

Basically Stacks.

In [ ]:
import threading
from queue import LifoQueue
import time

def mySubscriber(queue: LifoQueue):
    while not queue.empty():
        item = queue.get()
        if item is None:
            break
        print("{} removed {} from the queue".format(threading.current_thread(), item))
        queue.task_done()
        time.sleep(1)

myQueue = LifoQueue()

for i in range(10):
    myQueue.put(i)
print("Queue Populated")

threads = []
for i in range(2):
    thread = threading.Thread(target=mySubscriber, args=(myQueue,))
    thread.start()
    threads.append(thread)
    
for thread in threads:
    thread.join()
print("Queue is empty")

### 3. Priority Queues

With PriorityQueue, we can give everything that we put into the queue a weight as to how important it is. We can populate our PriorityQueues in much the same way that we
populate our normal queue object except that we use tuples, and pass in
priority_number as the first value in our tuple: (priority_number, data).

In [ ]:
import threading
from queue import PriorityQueue
import time

def mySubscriber(queue: PriorityQueue):
    while not queue.empty():
        item = queue.get()
        if item is None:
            break
        print("{} removed {} from the queue".format(threading.current_thread(), item))
        queue.task_done()
        time.sleep(1)

myQueue = PriorityQueue()

for i in range(5):
    myQueue.put(i, i)

for i in range(5):
    myQueue.put(i, i)

print("Queue Populated")
threads = []
for i in range(2):
    thread = threading.Thread(target=mySubscriber, args=(myQueue,))
    thread.start()
    threads.append(thread)

for thread in threads:
    thread.join()
    
print("Queue is empty")